## 英語音声のアクセントを分類するモデルを構築してみよう

あらかじめVCTK corpusを用意してください。

想定しているディレクトリ構造は以下の通りです。異なる場合は都度調整してください。

```
/~~/englishai/data/
├── csv/  
├── datasets/  
│   └── VCTK-Corpus-0.92/  
│       ├── speaker-info.txt # VCTKの元のスピーカー情報テキスト  
│       └── wav48_silence_trimmed/  # VCTKの.flacファイルが格納されたフォルダ  
│           ├── p225/  
│           │   ├── p225_001_mic1.flac  
│           │   ├── p225_002_mic1.flac  
│           │   └── …  
│           ├── p226/  
│           │   ├── p226_001_mic1.flac  
│           │   └── …  
│           └── …            # その他のスピーカーのフォルダ  
└── models/ 

```

コードを実行すると以下のようにファイルを配置します。
```
/~~/englishai/data/
├── csv/  
│   ├── vctkinfo.csv         # VCTKのスピーカー情報テキストを変換したCSVファイル  
│   └── vctk_data.csv        # 各音声ファイルのパスとラベル情報が記載されたCSVファイル  
├── datasets/  
│   └── VCTK-Corpus-0.92/  
│       ├── speaker-info.txt # VCTKの元のスピーカー情報テキスト  
│       └── wav48_silence_trimmed/  # VCTKの.flacファイルが格納されたフォルダ  
│           ├── p225/  
│           │   ├── p225_001_mic1.flac  
│           │   ├── p225_002_mic1.flac  
│           │   └── …  
│           ├── p226/  
│           │   ├── p226_001_mic1.flac  
│           │   └── …  
│           └── …            # その他のスピーカーのフォルダ  
└── models/  
    └── best_model.pth   # 学習済みモデルの重みファイル

```

このコードを実行するとVCTKコーパスのアクセント情報をもとに音声を分類するモデルを構築できます。

アクセント情報をラベルとして分類しますが、このデータ量では本当にアクセントをもとに分類できている可能性は**低い**です。(機械学習を用いずに音声を分析して、さまざまな音響特徴量を比較してクラスタリングしてみるとモデルの判断根拠を可視化できるかもしれません)

モデルを**改変**したり、**データを増やして学習**させてみて、音声を用いて実際に機械学習モデルの実装の練習をしてみてください。


In [1]:
from torchaudio.datasets import VCTK_092
from core.lstm_base import *

In [ ]:
config.base_data_path = "/User/~~/englishai/data" # ~~の部分に環境に合わせてパスを追加
config.update_paths()
data = VCTK_092(config.datasets_path, url='https://datashare.is.ed.ac.uk/bitstream/handle/10283/3443/VCTK-Corpus-0.92.zip', download=True)

input_file = os.path.join(config.vctk_dataset_path, "speaker-info.txt")
convert_vctkinfo(input_file, config.vctkinfo_csv)
process_all_data(data, config.vctkinfo_csv, config.vctk_data_csv)

In [2]:
# 学習
config.num_epochs = 25 # 学習epoch数
config.update_paths() # パラメータ更新
execute_pipeline(config.vctk_data_csv) # 学習を開始(高性能cpuでの演算時、25epochで1時間程度の計算量)

Epoch 1/25: Train Loss: 1.5981, Train Acc: 0.4349 | Val Loss: 1.3159, Val Acc: 0.5236
Epoch 2/25: Train Loss: 1.0272, Train Acc: 0.6203 | Val Loss: 0.7368, Val Acc: 0.7249
Epoch 3/25: Train Loss: 0.5899, Train Acc: 0.7890 | Val Loss: 0.5273, Val Acc: 0.8133
Epoch 4/25: Train Loss: 0.3749, Train Acc: 0.8707 | Val Loss: 0.4689, Val Acc: 0.8500
Epoch 5/25: Train Loss: 0.2582, Train Acc: 0.9134 | Val Loss: 0.2240, Val Acc: 0.9259
Epoch 6/25: Train Loss: 0.1853, Train Acc: 0.9380 | Val Loss: 0.2011, Val Acc: 0.9282
Epoch 7/25: Train Loss: 0.1688, Train Acc: 0.9444 | Val Loss: 0.1257, Val Acc: 0.9603
Epoch 8/25: Train Loss: 0.1199, Train Acc: 0.9609 | Val Loss: 0.1320, Val Acc: 0.9542
Epoch 9/25: Train Loss: 0.0921, Train Acc: 0.9701 | Val Loss: 0.0939, Val Acc: 0.9720
Epoch 10/25: Train Loss: 0.0888, Train Acc: 0.9715 | Val Loss: 0.1403, Val Acc: 0.9578
Epoch 11/25: Train Loss: 0.0836, Train Acc: 0.9733 | Val Loss: 0.0788, Val Acc: 0.9754
Epoch 12/25: Train Loss: 0.0605, Train Acc: 0.9803 |

In [3]:
# 推論
# 使用するcsv（学習時に用いたものと同じ）
csv_path = config.vctk_data_csv
test_audio_file = f"{config.wav_path}/p225/p225_001_mic1.flac"
model_path = config.model_file
inference(test_audio_file, model_path, csv_path)

推論結果: /speechai/data/datasets/VCTK-Corpus-0.92/wav48_silence_trimmed/p225/p225_001_mic1.flac のラベルは 'Scottish' です。


'Scottish'